# TakeOff.ai — Train the spaces (room) model on **Kaggle**

Why Kaggle: **Save & Run All (Commit)** runs this whole notebook *server-side to
completion* — close the tab, lose your wifi, doesn't matter. It finishes and
saves `best.pt` as downloadable output. ~30 GPU-hrs/week, 9 h per run.

### Before you run — 3 settings (right-hand panel)
1. **Accelerator -> GPU T4 x2** (or P100)
2. **Internet -> On**  (required for the clone + dataset download)
3. **Environment -> Always use latest**

### To run: don't run cells one-by-one — click **Save Version -> Save & Run All (Commit)**.
It runs in the background; come back when it's green and download `best.pt` from the
version's **Output** tab.

## 1. Confirm GPU + Internet

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Accelerator -> GPU in the right panel'
import urllib.request
try:
    urllib.request.urlopen('https://github.com', timeout=5); print('Internet: ON')
except Exception as e:
    print('Internet: OFF — turn it ON in the right panel. (', e, ')')

## 2. Get the code
The repo is private, so this needs a GitHub token with **read** access.

**Recommended:** add it as a Kaggle Secret so it's never written into the notebook:
*Add-ons -> Secrets -> + Add a new secret*, Label = `GITHUB_TOKEN`, value = your PAT,
then tick the checkbox to attach it. The cell below reads it automatically.

In [ ]:
import os
TOKEN = os.environ.get('GITHUB_TOKEN')
if not TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN')
    except Exception:
        TOKEN = ''  # public-clone fallback (will fail if repo is private)

REPO = 'github.com/Siddartha-DevOps/TakeOff.git'
url = f'https://{TOKEN}@{REPO}' if TOKEN else f'https://{REPO}'

%cd /kaggle/working
!rm -rf TakeOff
!git clone --depth 1 $url
%cd /kaggle/working/TakeOff/app/backend
print('cwd is now the backend package')

## 3. Install the ML stack

In [ ]:
%cd /kaggle/working/TakeOff/app/backend
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements-ml.txt
!python -m ml.preflight

## 4. Dataset — CubiCasa5K -> versioned YOLO-seg
~5 GB download + convert. Takes a few minutes; it's the same step that printed
`{'train': 3945, 'val': 987, ...}` for you before.

In [ ]:
%cd /kaggle/working/TakeOff/app/backend
import datetime
TS = datetime.datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
!python -c "from ml.datasets.acquire_cubicasa import download_cubicasa; download_cubicasa('cubicasa5k.zip')"
!mkdir -p data/cubicasa5k && unzip -q -o cubicasa5k.zip -d data/cubicasa5k
!python -m ml.datasets.acquire_cubicasa --root data/cubicasa5k --out data/spaces_v1 --created-at $TS
!python -m ml.preflight --data data/spaces_v1/data.yaml --require train

## 4b. Label sanity check — LOOK before you train
Near-zero training mAP almost always means the labels don't line up with the
pixels, not that the model needs longer. This draws each room's label polygon
back onto its image. **Open the images below (also saved to the Output tab):**
- Outlines sit *on* the rooms -> labels are good; training config is the lever.
- Outlines shifted / scaled / clustered in a corner -> the dataset conversion is
  the bug, and no amount of training fixes it. Stop and tell me what you see.

In [ ]:
%cd /kaggle/working/TakeOff/app/backend
!python -m ml.datasets.preview_labels --dataset data/spaces_v1 --split train --n 8 --out /kaggle/working/label_previews
import glob
from IPython.display import Image, display
for p in sorted(glob.glob('/kaggle/working/label_previews/*.png'))[:4]:
    print(p)
    display(Image(filename=p))

## 5. Smoke run (1 epoch) — proves the pipeline before the long train

In [ ]:
%cd /kaggle/working/TakeOff/app/backend
!python -m ml.training.run_training --data data/spaces_v1/data.yaml --task spaces --smoke --no-promote

## 6. Full training
`EPOCHS=100, IMGSZ=1280` are the tuned defaults — floor plans have thin walls and
many rooms, so the resolution matters (a smaller imgsz was the main reason an
earlier run scored near-zero mAP). This is close to a full Kaggle GPU window on a
T4; if the previews in 4b looked wrong, **don't run this** — fix the labels first.
Writes weights to `models/best.pt`.

In [ ]:
%cd /kaggle/working/TakeOff/app/backend
EPOCHS, IMGSZ = 100, 1280
!python -m ml.training.run_training --data data/spaces_v1/data.yaml --task spaces --epochs {EPOCHS} --imgsz {IMGSZ}

## 7. Prove accuracy (the promotion gate: mIoU >= 0.70, mAP@0.5 >= 0.50, err <= 5%)

In [ ]:
%cd /kaggle/working/TakeOff/app/backend
!python -m ml.eval.predict_golden --dataset data/spaces_v1 --weights models/best.pt --evaluate --out preds.json || true
!python -m ml.eval.build_golden --dataset data/spaces_v1 --out golden.json --predictions preds.json
!python -m ml.eval.report --golden golden.json --out accuracy_report.md || true
print(open('accuracy_report.md').read())

## 8. Save `best.pt` as notebook output
Copies the weights + accuracy report to `/kaggle/working` root so they appear in
this version's **Output** tab — download from there and drop into the backend at
`app/backend/models/best.pt`.

In [ ]:
import shutil, os
os.makedirs('/kaggle/working', exist_ok=True)
src = '/kaggle/working/TakeOff/app/backend/models/best.pt'
if os.path.exists(src):
    shutil.copy(src, '/kaggle/working/best.pt')
    for f in ('accuracy_report.md',):
        p = f'/kaggle/working/TakeOff/app/backend/{f}'
        if os.path.exists(p):
            shutil.copy(p, f'/kaggle/working/{f}')
    print('Saved -> /kaggle/working/best.pt', os.path.getsize('/kaggle/working/best.pt'), 'bytes')
    print('Download it from the Output tab of this version.')
else:
    print('best.pt not found — training did not complete. Check the Cell 6 log.')